# Step 2. The trial before

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter-template/blob/main/tutorial/notebooks/2_sequence.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 1 gave a block with half of the trials congruent: the ink colour crossed with congruency. This step keeps that block and adds one factor.

**The question.** Does a conflict change how the next trial is handled?

**Prediction (conflict adaptation).** The congruency effect is smaller after an incongruent trial than after a congruent one.

**The alternative to rule out.** Nothing is adjusted; trials that repeat a colour or a word from the trial before are simply faster, and such repeats are spread unevenly.

**The analysis.** The congruency effect after a congruent trial against the congruency effect after an incongruent trial.

### As in step 1

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, MinimumTrials,
                      CrossBlock, synthesize_trials, tabulate_experiments, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

In [ ]:
# a derived factor: it follows from colour and word on the same trial
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


congruency = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word])),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word])),
])

### New: the trial before as a factor

A transition factor follows from the trial before: `[0]` is this trial, `[-1]` the one before.

In [ ]:
def after_congruent(colour, word):
    return colour[-1] == word[-1]


def after_incongruent(colour, word):
    return colour[-1] != word[-1]


trial_before = Factor("trial_before", [
    DerivedLevel("congruent", Transition(after_congruent, [colour, word])),
    DerivedLevel("incongruent", Transition(after_incongruent, [colour, word])),
])

### The block

The crossing of step 1 with the trial before added: 12 combinations, two rounds, plus a first trial, which has no trial before.

In [ ]:
block = CrossBlock(design=[colour, word, congruency, trial_before],
                   crossing=[colour, congruency, trial_before],
                   constraints=[MinimumTrials(25)])
trials = pd.DataFrame(synthesize_trials(block, 1, CMSGen)[0])
scored = trials[trials["trial_before"] != ""]        # without the first trial
display(pd.crosstab(scored["trial_before"], scored["congruency"]))

congruency,congruent,incongruent
trial_before,,
congruent,6,6
incongruent,6,6


The four combinations are equal, and that is by design: run the cell again and they stay equal.

### What else repeats?

Does the ink colour, and so the response, repeat from the trial before? That is a transition factor too. Put it in the design but not in the crossing: SweetPea then records it and leaves it free. Count it in each of the four combinations.

In [ ]:
def response_repeats(colour):
    return colour[0] == colour[-1]


def response_changes(colour):
    return colour[0] != colour[-1]


response = Factor("response", [
    DerivedLevel("repeats", Transition(response_repeats, [colour])),
    DerivedLevel("changes", Transition(response_changes, [colour])),
])

block = CrossBlock(design=[colour, word, congruency, trial_before, response],
                   crossing=[colour, congruency, trial_before],
                   constraints=[MinimumTrials(25)])
trials = pd.DataFrame(synthesize_trials(block, 1, CMSGen)[0])
scored = trials[trials["trial_before"] != ""]
display(pd.crosstab([scored["trial_before"], scored["congruency"]], scored["response"]))

response                  changes  repeats
trial_before congruency                   
congruent    congruent          5        1
             incongruent        5        1
incongruent  congruent          4        2
             incongruent        6        0

**Check.** Is the response repeat equal across the four combinations? If not, the alternative explanation is still open. Whatever is not in the crossing is left to chance: run the block cell again and see which counts change.

**Try.** Ask what it would take to forbid all repeats (`Exclude` on a transition factor for the colour), and what that does to the four combinations.

**Next.** Step 3 goes from the trial before to the whole block: mostly congruent and mostly incongruent blocks.